# Summary

## Tình huống

1.  Ứng Dụng: GrabMart
    - Chương trình khuyến mãi: Cross-sell mega promo (chương trình khuyến mãi chéo quy mô lớn)
    - Khoảng tgian thi triển: 6 tuần
    - Địa diểm: Thành Phố Hồ Chí Minh
    - Trạng Thái: thí điểm

2. Merchanism:
    Khi đặt đơn hàng trên `GrabFood`:
    constraints:
        - tổng giá trị > 100.000 vnđ -> nhận voucher trị giá 50.000 vnđ cho `GrabMart` tiếp theo

3. Output - Backlog Test:
    - Tổng giá trị giao dịch (GMV):
        + GrabMart và GrabFood: đều tăng hơn 45% (so với ?)

4. Deployment:
    - Triển khai chính thức tại Hà Nội (với ngân sách khủng)

5. Work-to-do: đưa ra đánh giá -- và phân tích răng có hợp lí để deploy ở Hà Nội không ?

    Tình huống:

        Giám đốc Quốc gia tin rằng chiến dịch này đã thành công trong việc thay đổi hành vi người dùng và 
        hình thành thói quen sử dụng đa dịch vụ.

    Câu hỏi:
        
        - Dựa trên dữ liệu chuyên sâu về hành vi và kết quả kinh doanh (P&L), bạn có đồng ý với nhận định này không?
        - Yếu tố thực sự nào đang thúc đẩy mức tăng trưởng 45% đó? 
        - Hãy xây dựng lập luận dựa trên dữ liệu và đưa ra khuyến nghị cuối cùng về việc có nên mở rộng chương trình này sang Hà Nội hay không 

## Import Libraries

In [14]:
import sqlite3
import pandas as pd

## Create Connection to Database

In [15]:
conn = sqlite3.connect("grab_vn_case_analytics.db")

## Analytics view

### View table available

In [16]:
query = """
SELECT name, sql
FROM sqlite_master
WHERE type = 'table';
"""

for name, sql in conn.execute(query):
    print(name)
    print(sql)

users
CREATE TABLE "users" (
"user_id" TEXT,
  "city" TEXT,
  "user_segment" TEXT,
  "first_mart_order_date" TEXT
)
merchants
CREATE TABLE "merchants" (
"merchant_id" TEXT,
  "merchant_name" TEXT,
  "vertical" TEXT,
  "merchant_type" TEXT,
  "take_rate" REAL
)
weather_daily
CREATE TABLE "weather_daily" (
"date" TEXT,
  "city" TEXT,
  "avg_temperature_celsius" REAL
)
transactions
CREATE TABLE "transactions" (
"transaction_id" TEXT,
  "date" TEXT,
  "week_number" INTEGER,
  "period" TEXT,
  "city" TEXT,
  "service_type" TEXT,
  "user_id" TEXT,
  "merchant_id" TEXT,
  "item_subtotal" REAL,
  "delivery_fee_collected" REAL,
  "driver_payout" REAL,
  "promo_discount" REAL,
  "is_cross_sell_voucher" INTEGER
)


### Prepare dataframe

### Deep-Analysis -- with query

In [17]:
class SQLite3Connection:
    def __init__(
        self,
        db_path:str
    ):
        self.db_path = db_path
        self.conn = sqlite3.connect(self.db_path)

    @staticmethod
    def parse_query(
        query_path:str
    ) -> str:
        with open(query_path, 'r') as query:
            return query.read()

    def query_to_dataframe(
        self,
        query:str
    ) -> pd.DataFrame:
        cursor = self.conn.execute(query)
        data = cursor.fetchall()
        header = [col[0] for col in cursor.description]

        return pd.DataFrame(data=data, columns=header)

In [18]:
data = SQLite3Connection(db_path="grab_vn_case_analytics.db")

## Background-Analysis

## In-Promotion Duration

### Acquisition Funnel

`query_path`: [acquisition_funnel](query/acquisition_funnel.sql)

**Hypothesis:**

`45% GMV growth` của Food có thể được campaign kích hoạt từ existing Food users -- vì mechanic bắt đầu từ một `Food order >100k`

```mermaid
flowchart TD
    A[Existing GrabFood user] -->|Food order > 100k| B[Eligible for voucher]
    B -->|50k voucher| C[GrabMart voucher redemption]
    C --> D[GrabMart transaction]
```

| Metric                            | Ý nghĩa                                                 |
| --------------------------------- | ------------------------------------------------------- |
| **Food eligibility rate**         | % Food users/orders đủ điều kiện                        |
| **Voucher redemption rate**       | % eligible users thực sự redeem voucher                 |
| **Mart acquisition rate**         | % Food users được campaign chuyển thành Mart users      |
| **Voucher-driven Mart GMV share** | Mart GMV đến từ transaction có voucher / total Mart GMV |
| **Voucher-driven GMV uplift**     | incremental GMV nếu có thể establish baseline/control   |


_result:_ 



| |metric | value |
--|-------|-------|
0|food_users|4999.00|
1|eligible_food_users|4991.00|
2|voucher_redeemers|4446.00|
3|campaign_acquired_users|4441.00|
4|food_eligibility_rate|99.84|
5|voucher_redemption_rate|89.08|
6|campaign_acquisition_rate|88.84|





In [19]:
query=data.parse_query(query_path="query/acquisition_funnel.sql")
df = data.query_to_dataframe(query=query)
value_arr = df.columns
df_pivoted = df.melt(
    value_vars=value_arr,
    var_name="metric",
    value_name="value"
)

---
### % voucher-driven transaction int Total-GMV
---
>Trong 45% growth của GMV (both Mart/Food) thì bao nhiêu phần trăm đến từ voucher của Mart

`sql_path`: [percentage-voucher-driven-transaction](query/percentage_voucher_driven_transaction.sql)

_result_:


| |metric|value|
|-|------|-----|
0|total_mart_gmv|1792132.50
1|voucher_mart_gmv|781755.20
2|voucher_gmv_share|43.62





In [20]:
query=data.parse_query(query_path="query/percentage_voucher_driven_transaction.sql")
df=data.query_to_dataframe(query=query)
value_arr = df.columns
df_pivoted = df.melt(
    value_vars=value_arr,
    var_name="metric",
    value_name="value"
)

---
### Acquisition-Funnel (Deeper Layer)
---
>Trong số user/GMV của GrabMart trong Promo period, campaign thực sự mang vào bao nhiêu user mới so với những người vốn đã là Mart user?

Structure song song:

```
Promo-period GrabMart users
│
├── Existing Mart User
│   └── đã từng dùng Mart trước campaign
│
└── New Mart User
    └── first_mart_order_date bắt đầu trong/đúng thời điểm promo
```
sau đó -- `New-Mart` sẽ tiếp tục spill:
```
New Mart Users
│
├── Cross-sell acquired
│   └── is_cross_sell_voucher = 1
│
└── Non-cross-sell
```
`sql_path`: [acquisition-funnel-inc-user-type](query/acquisition_funnel_inc_user_type.sql)


__Result__:


| mart_user_type | users | user_share | cross_sell_users | cross_sell_user_rate | mart_gmv | gmv_share | cross_sell_gmv | cross_sell_gmv_share | gmv_per_user |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| New Mart User | 643 | 13.24 | 643 | 100.00 | 240,164.6 | 13.4 | 111,391.1 | 46.38 | 373.51 |
| Existing Mart User | 4,215 | 86.76 | 3,804 | 90.25 | 1,551,967.9 | 86.6 | 670,364.1 | 43.19 | 368.20 |

In [21]:
query = data.parse_query(query_path="query/acquisition_funnel_inc_user_type.sql")
data.query_to_dataframe(query=query)


,mart_user_type,users,user_share,cross_sell_users,cross_sell_user_rate,mart_gmv,gmv_share,cross_sell_gmv,cross_sell_gmv_share,gmv_per_user
0,New Mart User,643,13.24,643,100.00,240164.6,13.4,111391.1,46.38,373.51
1,Existing Mart User,4215,86.76,3804,90.25,1551967.9,86.6,670364.1,43.19,368.20


## Finding#1:

```bash
Campaign
   │
   ├── Voucher adoption: HIGH
   │       89.08% Food users
   │
   ├── New Mart acquisition: EXISTS
   │       643 users
   │       13.24% Mart users
   │
   └── Existing Mart users:
           86.76% Mart users
           43.19% GMV cross-sell share
```

Hypothesis hình thành là:

>Campaign clearly generated cross-sell activity and acquired a measurable group of new Mart users, but the majority of Promo-period Mart users and GMV still came from existing Mart users.

Đặc biệt mình muốn kiểm tra 2 things:
- 643 New Mart Users này đóng góp bao nhiêu vào incremental growth so với baseline?
- Sau khi campaign kết thúc ở Week 13–14, 643 users này có tiếp tục dùng Mart không?

> Nếu họ tiếp tục → evidence cho habit formation.
>
> Nếu họ biến mất ngay sau Promo → evidence cho promo-driven temporary acquisition.


---
### Remarkable the Storyline until we get things up

--- 

```bash

                    45% GMV Growth
                          │
                          ▼
                 ┌─────────────────┐
                 │ Cross-sell works │
                 └────────┬────────┘
                          │
                    643 New Mart     <---- We are here :"> hehe
                       13.24%
                          │
             ┌────────────┴────────────┐
             │                         │
       How much did they          Do they retain?
       contribute to uplift?      Week 13–14?
             │                         │
             ▼                         ▼
       Growth attribution       Habit formation
             │                         │
             └────────────┬────────────┘
                          ▼
                Is 45% sustainable?
                          │
                          ▼
                   Hanoi decision

```

---
### How much New Mart contribute to the uplift -- Test in this cohort

---

1. Phân rã Uplift theo tệp người dùng (user-segmentation) (Done)
2. Phân rã theo hành vi giỏ hàng (basket behavior and gaming)

    **- Hành vi kéo giỏ hàng: trung bình đơn Food trước đây của người dùng là 70-80k. Nhờ promo, họ cố mua thêm để bill nhảy lên mức > 100k. Giải thích được vì sao GMV cũng tăng**

    `query_path`: [basket-behavior](query/basket_behavior.sql)

    _Result_:

    | Period | Total Food Orders | AOV (VND '000) | < 80K (%) | 80K - 99.9K (%) | 100K - 119.9K (%) | >= 120K (%) |
    | --- | --- | --- | --- | --- | --- | --- |
    | **Pre-Promo** | 4,056 | 133.8 | 11.8 | 14.6 | 19.7 | 53.9 |
    | **Promo** | 9,555 | 115.4 | 8.1 | 9.9 | 50.7 | 31.3 |

    
    **- Hành vi của cùng 1 tệp khách hàng (trước và trong khi diễn ra promo)**

    `query_path`: [same_user_behavior](query/same_user_behavior.sql)

    _Result_:

    | Total Tracked Users | User AOV Pre-Promo | User AOV During Promo | Avg Uplift Per User | Users Crossing Threshold (%) |
    | --- | --- | --- | --- | --- |
    | 4,985 | 130.2 | 118.2 | -12.0 | 5.8 |

    
    **- Hành vi chẻ đơn: Người dùng muốn đặt 300k, thay vì đặt 1 đơn -- thì chia làm 3 để lấy 3 vouchers. GMV Food tăng về số lượng đơn nhưng chi phí khuyến mãi (Promo cost) bị nhân lên gấp 3 --> loss in PnL**

    `query_path`: [order-splitting](query/order_splitting.sql)

    _Result_:
    Metric | Pre-Promo | Promo | % Change |
    | --- | --- | --- | --- |
    | **Cohort Size** | 4,985 | 4,985 | - |
    | **Avg Orders per User** | 8.14 | 19.15 | **+135.3%** |
    | **Avg GMV per User** | 1,088.9 | 2,210.0 | **+103.0%** |
    | **Avg GMV per Order (AOV)** | 130.2 | 118.2 | **-9.2%** |
    | **Avg Qualification Order per User**| 5.98 | 15.67 | **+162%**|


In [28]:
# A. Basket pulling /threshold gaming
query = data.parse_query(query_path='query/basket_behavior.sql')
data.query_to_dataframe(query=query)


,period,total_food_orders,aov_vnd_k,shared_below_80k,shared_80k_to_99k,shared_100k_to_119k,shared_above_120k
0,Pre-Promo,40564,133.8,11.8,14.6,19.7,53.9
1,Promo,95552,115.4,8.1,9.9,50.7,31.4


In [32]:
# B. same-user behaviour
query = data.parse_query(query_path="query/same_user_behavior.sql")
data.query_to_dataframe(query=query)


,total_tracked_users,user_aov_pre_promo,user_aov_during_promo,avg_uplift_per_user,percentage_users_crossing_threshold
0,4985,130.2,118.2,-12.0,5.8


In [34]:
# C. Order splitting / voucher gaming
query = data.parse_query(query_path="query/order_splitting.sql")
data.query_to_dataframe(query=query)


,cohort_size,avg_orders_per_user_pre,avg_qualifying_orders_pre,avg_gmv_per_user_pre,avg_gmv_per_order_pre,avg_orders_per_user_promo,avg_qualifying_orders_promo,avg_gmv_per_user_promo,avg_gmv_per_order_promo,pct_change_orders_per_user,pct_change_qualifying_orders,pct_change_gmv_per_user,pct_change_gmv_per_order
0,4985,8.14,5.98,1088.9,130.2,19.15,15.67,2210.0,118.2,135.3,162.0,103.0,-9.2


### PnL support Pre-Promo and Promo

`query_path`: [pnl-prepromo-promo](query/pnl_prepromo_promo.sql) 

**Assumption PnL Calculation**:
```bash
Gross Revenue
= item_subtotal × merchant take_rate
  + delivery_fee_collected

Direct Cost
= driver_payout + promo_discount

Contribution Margin
= Gross Revenue - Direct Cost
```

### Order Splitting & P&L Impact (Tác động P&L của hành vi chẻ đơn)

| Service | Period | Total Orders | Total GMV (VND '000) | Avg Revenue / Order | Avg Cost / Order | Avg Margin / Order | Total Contribution Margin (VND '000) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| **GrabFood** | Pre-Promo | 40,564 | 5,428,398.4 | 31.8 | 23.0 | **8.8** | **357,110.7** |
| **GrabFood** | Promo | 95,552 | 11,030,020.5 | 29.0 | 23.7 | **5.3** | **506,458.8** |
|  |  |  |  |  |  |  |  |
| **GrabMart** | Pre-Promo | 14,517 | 1,383,717.7 | 24.5 | 20.0 | **4.5** | **64,848.0** |
| **GrabMart** | Promo | 23,630 | 1,792,132.5 | 22.6 | 47.6 | **-25.0** | **-590,574.8** |
|  |  |  |  |  |  |  |  |
| **TOTAL** | **Pre-Promo** | 55,081 | 6,812,116.1 | - | - | - | **421,958.7** |
| **TOTAL** | **Promo** | 119,182 | 12,822,153.0 | - | - | - | **-84,116.0** |

---

**Nhận xét nhanh đi kèm bảng:**

* **Top-line (Bề nổi):** Tốc độ tăng trưởng số lượng đơn và GMV cực kỳ ấn tượng, vượt xa mốc 45% (Food GMV x2, Mart GMV +30%).
* **Bottom-line (Sự thật):** Biên lợi nhuận trung bình (`Avg Margin / Order`) của Food sụt giảm 40% do hành vi chẻ đơn làm giảm doanh thu trên mỗi cuốc (29.0 vs 31.8).
* **Vết thương chí mạng:** Mart gánh chi phí Voucher khổng lồ (Cost vọt từ 20.0 lên 47.6), khiến mỗi đơn Mart kích hoạt bằng Promo đều **âm 25k**, kéo sập tổng P&L của cả hệ sinh thái từ mức lời hơn 400 triệu xuống mức **âm 84 triệu VND**.

In [36]:
# C. Order splitting / voucher gaming
query = data.parse_query(query_path="query/pnl_prepromo_promo.sql")
data.query_to_dataframe(query=query)


,service_type,period,total_orders,total_gmv,avg_revenue_per_order,avg_cost_per_order,avg_margin_per_order,total_contribution_margin
0,GrabFood,Pre-Promo,40564,5428398.4,31.8,23.0,8.8,357110.7
1,GrabFood,Promo,95552,11030020.5,29.0,23.7,5.3,506458.8
2,GrabMart,Pre-Promo,14517,1383717.7,24.5,20.0,4.5,64848.0
3,GrabMart,Promo,23630,1792132.5,22.6,47.6,-25.0,-590574.8


### Query-base (DRAFT)

### Check-whether: in Promo Duration -- is there any user | not aware about the promotion

_no_used_any_transaction_in_promo_duration_

`sql_path`: query/user_no_aware_promotion.sql

user_segment | cnt_user | Food_users | Mart_users
-------------|----------|------------|-----------
High Value|29|29|29
Occasional|288|288|167
Regular|168|168|152
Voucher Hunter|68|68|63

In [22]:
query=data.parse_query(query_path="query/user_no_aware_promotion.sql")
data.query_to_dataframe(query=query)

,user_segment,cnt_user,Food_users,Mart_users
0,High Value,29,29,29
1,Occasional,288,288,167
2,Regular,168,168,152
3,Voucher Hunter,68,68,63


### Segment (existed | new) in each duration (Before-Promo | Promo | After-Promo) 

`sql_path`: "query/classfiy_users_new_new_or_exists.sql"

||period|user_type|unique_users|n_transactions
|-|------|---------|------------|--------------
|0|Post-Promo|existed-mart-user|2634|4225
|1|Pre-Promo|existed-mart-user|3984|12543
|2|Promo|existed-mart-user|4215|20420
|3|Post-Promo|new-mart-user|385|655
|4|Pre-Promo|new-mart-user|598|1974
|5|Promo|new-mart-user|643|3210

In [23]:
query=data.parse_query(query_path="query/classify_users_new_or_exist.sql")
data.query_to_dataframe(query=query)

,period,user_type,unique_users,n_transactions
0,Post-Promo,existed-mart-user,2634,4225
1,Pre-Promo,existed-mart-user,3984,12543
2,Promo,existed-mart-user,4215,20420
3,Post-Promo,new-mart-user,385,655
4,Pre-Promo,new-mart-user,598,1974
5,Promo,new-mart-user,643,3210


### who actual drove the GROWTH ?

In [24]:
query=data.parse_query(query_path="query/segment_drive_growth.sql")
data.query_to_dataframe(query=query)

,city,user_segment,user_type,gross_gmv_M_vnd,service_type
0,Ho Chi Minh City,High Value,existed-mart-user,4472.2,GrabFood
1,Ho Chi Minh City,Regular,existed-mart-user,2950.7,GrabFood
2,Ho Chi Minh City,Voucher Hunter,existed-mart-user,1558.7,GrabFood
3,Ho Chi Minh City,High Value,new-mart-user,795.6,GrabFood
4,Ho Chi Minh City,Regular,existed-mart-user,734.4,GrabMart
5,Ho Chi Minh City,Occasional,existed-mart-user,509.7,GrabFood
6,Ho Chi Minh City,Regular,new-mart-user,441.0,GrabFood
7,Ho Chi Minh City,High Value,existed-mart-user,406.7,GrabMart
8,Ho Chi Minh City,Voucher Hunter,existed-mart-user,318.7,GrabMart
9,Ho Chi Minh City,Voucher Hunter,new-mart-user,254.0,GrabFood


### Check PnL: structure

In [25]:
query = """
select
city,
user_segment,
count(user_id) as cnt_user
from users
where 1=1
group by
    city,
    user_segment
"""
data.query_to_dataframe(query=query)

,city,user_segment,cnt_user
0,Da Nang,High Value,408
1,Da Nang,Occasional,307
2,Da Nang,Regular,919
3,Da Nang,Voucher Hunter,366
4,Hanoi,High Value,738
5,Hanoi,Occasional,489
6,Hanoi,Regular,1604
7,Hanoi,Voucher Hunter,669
8,Ho Chi Minh City,High Value,984
9,Ho Chi Minh City,Occasional,730
